# 📋 Notebook 5 — Final Summary Dashboard
**Project:** Customer Churn Prediction  
**Author:** Aditya Bobade

---
Generates the executive summary dashboard chart and statistical summary CSV.


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

original = pd.read_csv('../data/telco_churn.csv')
risk_df  = pd.read_csv('../data/telco_churn_risk_scored.csv')
tier_order  = ['Low Risk','Medium Risk','High Risk','Critical Risk']
tier_colors = ['#2ecc71','#f39c12','#e74c3c','#8e44ad']
print("Loaded:", risk_df.shape)


Loaded: (7043, 23)


In [2]:
# === EXECUTIVE SUMMARY DASHBOARD ===
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#1a1a2e')
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

t_style = dict(color='white', fontsize=11, fontweight='bold', pad=8)

# KPI tiles (row 0)
kpis = [
    ("Total Customers",    f"{len(original):,}",  "#3498db"),
    ("Churn Rate",         f"{(original['Churn']=='Yes').mean()*100:.1f}%", "#e74c3c"),
    ("Avg Monthly Charge", f"${original['MonthlyCharges'].mean():.2f}", "#2ecc71"),
]
for idx, (lbl, val, col) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, idx])
    ax.set_facecolor(col)
    ax.text(0.5, 0.65, val, ha='center', va='center',
            fontsize=22, fontweight='bold', color='white', transform=ax.transAxes)
    ax.text(0.5, 0.25, lbl, ha='center', va='center',
            fontsize=9, color='white', transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])

# Churn by Contract
ax1 = fig.add_subplot(gs[1, 0]); ax1.set_facecolor('#16213e')
cc = original.groupby('Contract')['Churn'].apply(lambda x: (x=='Yes').mean()*100)
bars = ax1.bar(cc.index, cc.values, color=['#e74c3c','#f39c12','#2ecc71'], edgecolor='none')
ax1.set_title('Churn Rate by Contract', **t_style)
ax1.tick_params(colors='white', labelsize=8); ax1.spines[:].set_visible(False)
for bar, v in zip(bars, cc.values):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{v:.0f}%', ha='center', color='white', fontsize=8, fontweight='bold')

# Risk Tier Pie
ax2 = fig.add_subplot(gs[1, 1]); ax2.set_facecolor('#16213e')
tc = [len(risk_df[risk_df['RiskTier']==t]) for t in tier_order]
wedges, texts, autotexts = ax2.pie(tc, labels=tier_order, autopct='%1.1f%%',
    colors=tier_colors, startangle=90, wedgeprops=dict(edgecolor='#1a1a2e', linewidth=2))
for t in texts:     t.set_color('white'); t.set_fontsize(7)
for at in autotexts: at.set_color('white'); at.set_fontsize(7)
ax2.set_title('Risk Tier Split', **t_style)

# MonthlyCharges by Churn
ax3 = fig.add_subplot(gs[1, 2]); ax3.set_facecolor('#16213e')
for lbl, col in [('No','#2ecc71'),('Yes','#e74c3c')]:
    ax3.hist(original[original['Churn']==lbl]['MonthlyCharges'],
             bins=25, alpha=0.7, color=col, label=f'Churn: {lbl}', edgecolor='none')
ax3.set_title('Monthly Charges by Churn', **t_style)
ax3.tick_params(colors='white', labelsize=8); ax3.legend(fontsize=7, facecolor='#16213e', labelcolor='white')
ax3.spines[:].set_visible(False)

# Tenure vs RiskScore
ax4 = fig.add_subplot(gs[2, :2]); ax4.set_facecolor('#16213e')
cmap = {'Low Risk':'#2ecc71','Medium Risk':'#f39c12','High Risk':'#e74c3c','Critical Risk':'#8e44ad'}
for t in tier_order:
    sub = risk_df[risk_df['RiskTier']==t]
    ax4.scatter(sub['tenure'], sub['RiskScore'], c=cmap[t], alpha=0.3, s=10, label=t)
ax4.set_title('Tenure vs Churn Risk Score', **t_style)
ax4.set_xlabel('Tenure (months)', color='white', fontsize=9)
ax4.set_ylabel('Risk Score', color='white', fontsize=9)
ax4.tick_params(colors='white', labelsize=8); ax4.legend(fontsize=7, facecolor='#16213e', labelcolor='white', markerscale=2)
ax4.spines[:].set_visible(False)

# Churn by Internet Service
ax5 = fig.add_subplot(gs[2, 2]); ax5.set_facecolor('#16213e')
ic = original.groupby('InternetService')['Churn'].apply(lambda x: (x=='Yes').mean()*100).sort_values()
ax5.barh(ic.index, ic.values, color=['#2ecc71','#f39c12','#e74c3c'], edgecolor='none')
ax5.set_title('Churn by Internet Service', **t_style)
ax5.tick_params(colors='white', labelsize=8); ax5.spines[:].set_visible(False)
for i, v in enumerate(ic.values):
    ax5.text(v+0.3, i, f'{v:.1f}%', va='center', color='white', fontsize=8, fontweight='bold')

fig.suptitle('Customer Churn Prediction — Executive Dashboard',
             color='white', fontsize=16, fontweight='bold', y=0.98)
plt.savefig('../charts/14_final_summary_dashboard.png', bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print("Dashboard saved!")


Dashboard saved!


In [3]:
# Statistical Summary CSV
summary = pd.DataFrame({
    'Metric': [
        'Total Customers','Churned Customers','Churn Rate (%)',
        'Avg Tenure Churned (mo)','Avg Tenure Retained (mo)',
        'Avg Monthly Charge Churned ($)','Avg Monthly Charge Retained ($)',
        'Low Risk Customers','Medium Risk Customers',
        'High Risk Customers','Critical Risk Customers',
        'Monthly Revenue at Risk ($)',
    ],
    'Value': [
        len(original),
        (original['Churn']=='Yes').sum(),
        round((original['Churn']=='Yes').mean()*100,2),
        round(original[original['Churn']=='Yes']['tenure'].mean(),1),
        round(original[original['Churn']=='No']['tenure'].mean(),1),
        round(original[original['Churn']=='Yes']['MonthlyCharges'].mean(),2),
        round(original[original['Churn']=='No']['MonthlyCharges'].mean(),2),
        len(risk_df[risk_df['RiskTier']=='Low Risk']),
        len(risk_df[risk_df['RiskTier']=='Medium Risk']),
        len(risk_df[risk_df['RiskTier']=='High Risk']),
        len(risk_df[risk_df['RiskTier']=='Critical Risk']),
        round(risk_df[risk_df['RiskTier'].isin(['High Risk','Critical Risk'])]['MonthlyCharges'].sum(),2),
    ]
})
summary.to_csv('../charts/statistical_summary.csv', index=False)
print(summary.to_string(index=False))


                         Metric     Value
                Total Customers   7043.00
              Churned Customers   2032.00
                 Churn Rate (%)     28.85
        Avg Tenure Churned (mo)     25.40
       Avg Tenure Retained (mo)     34.50
 Avg Monthly Charge Churned ($)     72.54
Avg Monthly Charge Retained ($)     63.13
             Low Risk Customers   1946.00
          Medium Risk Customers   2041.00
            High Risk Customers   2390.00
        Critical Risk Customers    666.00
    Monthly Revenue at Risk ($) 229890.66
